In [11]:
import numpy as np
import networkx as nx
import pandas as pd
from pathlib import Path


In [12]:
def extract_data(file_path):
    file = Path(file_path)
    G = nx.read_graphml(file)

    node_num_cols = ["dn_position_x", "dn_position_y", "dn_position_z", "dn_correspondence_id"]
    edge_num_cols = ["fiber_length_mean", "FA_mean", "number_of_fibers"]

    node_records = []
    for node_id, attrs in G.nodes(data=True):
        node_records.append({"file": file.name, "path": str(file), "node_id": node_id, **attrs})
    nodes_df = pd.DataFrame(node_records)
    for col in node_num_cols:
        if col in nodes_df.columns:
            nodes_df[col] = pd.to_numeric(nodes_df[col], errors="coerce")

    edge_records = []
    for u, v, attrs in G.edges(data=True):
        edge_records.append({"file": file.name, "path": str(file), "source": u, "target": v, **attrs})
    edges_df = pd.DataFrame(edge_records)
    for col in edge_num_cols:
        if col in edges_df.columns:
            edges_df[col] = pd.to_numeric(edges_df[col], errors="coerce")

    return G, nodes_df, edges_df


In [13]:
file_path = Path("../data/123824_repeated10_scale250.graphml")

In [14]:
def setup_reservoir(
    file_path,
    lambda_reg=3e-6,
    t_thermalization=200,
    t_training=6000,
    sigma=10.0,
    rho=28.0,
    beta=8 / 3,
    dt=0.01,
    seed=42,
):
    G, nodes_df, edges_df = extract_data(file_path)

    N = G.number_of_nodes()
    node_ids = sorted(G.nodes())
    id_to_idx = {nid: i for i, nid in enumerate(node_ids)}

    W = np.zeros((N, N))
    for u, v, attrs in G.edges(data=True):
        i, j = id_to_idx[u], id_to_idx[v]
        w = float(attrs.get("number_of_fibers", 1.0))
        W[i, j] = w
        W[j, i] = w
    row_sums = W.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1
    W = W / row_sums

    def lorenz_euler(n_steps):
        data = np.empty((n_steps, 3))
        state = np.array([1.0, 1.0, 1.0])
        for i in range(n_steps):
            x, y, z = state
            state = state + dt * np.array([
                sigma * (y - x),
                x * (rho - z) - y,
                x * y - beta * z,
            ])
            data[i] = state
        return data

    total_steps = t_thermalization + t_training
    lorenz_full = lorenz_euler(total_steps)
    u_tr = lorenz_full[t_thermalization:]

    rng = np.random.default_rng(seed)
    W_in = rng.normal(0.0, 1.0 / np.sqrt(3), size=(N, 3))

    r = np.zeros(N)
    for t in range(t_thermalization):
        x, y, z = lorenz_full[t]
        r = np.tanh(W @ r + W_in @ np.array([x, y, z]))

    T = t_training
    R = np.empty((T, N))
    for t in range(T):
        r = np.tanh(W @ r + W_in @ u_tr[t])
        R[t] = r

    RtR = R.T @ R
    RtU = R.T @ u_tr
    W_out = np.linalg.solve(RtR + lambda_reg * np.eye(N), RtU)

    return W, W_in, W_out, r


In [15]:
def run_simulation(W, W_in, W_out, r_last, n_steps=100, dt=0.01, sigma=10.0, rho=10.0, beta=8 / 3):
    N = W.shape[0]
    traj = np.empty((n_steps, N))

    r = r_last.copy()

    for t in range(n_steps):
        u_hat = r @ W_out
        r = np.tanh(W @ r + W_in @ u_hat)
        traj[t] = r

    return traj


In [17]:
W, W_in, W_out, r_last = setup_reservoir(file_path)
traj = run_simulation(W, W_in, W_out, r_last)

# print(traj)